In [ ]:
import pandas as pd
from collections import Counter
import re

input_path = "RecipeDB_general.csv"          # path to your input CSV
output_path = "process_frequencies.csv"
process_col = "Processes"           # name of the processes column

# read only the processes column to save memory
df = pd.read_csv(input_path, usecols=[process_col], dtype={process_col: "string"}, encoding="utf-8")

counter = Counter()
split_re = re.compile(r"\|\||\|")   # split on '||' or '|' just in case

for raw in df[process_col].dropna():
    # normalize, split, strip, ignore empty tokens
    for token in split_re.split(str(raw)):
        t = token.strip().lower()
        if t:
            counter[t] += 1

# Convert to DataFrame sorted by frequency desc
out_df = pd.DataFrame(counter.items(), columns=["process", "frequency"]).sort_values(
    by="frequency", ascending=False
)

out_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved {len(out_df)} unique processes to {output_path}")

out_df = out_df.reset_index(drop=True)   
out_df.index = out_df.index + 1          
print(out_df.head(10))

Saved 270 unique processes to process_frequencies.csv
   process  frequency
1      add     187921
2     heat     102759
3     cook      94106
4     stir      83544
5    place      55651
6      mix      53863
7    cover      50143
8   remove      49561
9    serve      42398
10    boil      39022


In [3]:
import os
from pathlib import Path
import pandas as pd

input_csv = "RecipeDB_general.csv"
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)
output_csv = output_dir / "process_counts.csv"

# expected column names in your file
cols_wanted = ["Recipe_id", "total_time", "Recipe_title", "Processes"]

# read header first to decide which cols to load (safer for large files)
with open(input_csv, "r", encoding="utf-8", newline="") as f:
    header = f.readline().strip().split(",")

usecols = [c for c in cols_wanted if c in header]
df = pd.read_csv(input_csv, usecols=usecols, dtype=str, encoding="utf-8", low_memory=False)

# ensure all wanted columns exist
for c in cols_wanted:
    if c not in df.columns:
        df[c] = ""

# normalize Processes and count occurrences (count repeated entries)
def normalize_and_count(proc_str):
    if not proc_str or str(proc_str).strip() == "":
        return "", 0
    parts = [p.strip() for p in str(proc_str).split("||")]
    parts = [p for p in parts if p]  # drop empty
    return "||".join(parts), len(parts)

norm = df["Processes"].fillna("").astype(str).apply(normalize_and_count)
df["Processes"] = [t[0] for t in norm]
df["processes_count"] = [t[1] for t in norm]

out_cols = ["Recipe_id", "total_time", "Recipe_title", "Processes", "processes_count"]
df[out_cols].to_csv(output_csv, index=False, encoding="utf-8")

print(f"Saved {len(df)} rows to {output_csv}")

Saved 118083 rows to results\process_counts.csv


In [ ]:
import pandas as pd
import numpy as np
import os

# Path to your folder
base_path = "results"

# Load both CSVs
classified = pd.read_csv(os.path.join(base_path, "classified_processes_fixed.csv"))
recipes = pd.read_csv(os.path.join(base_path, "process_counts.csv"))

# Clean process names
classified['process'] = classified['process'].astype(str).str.strip().str.lower()
recipes['Processes'] = recipes['Processes'].astype(str).str.lower()

# Map NOVA labels to fixed numeric scores
nova_score_map = {
    'NOVA 1': 1.0,
    'NOVA 2': 1.5,
    'NOVA 3': 2.0,
    'NOVA 4': 4.0
}
classified['score'] = classified['nova_label'].map(nova_score_map)

# Create lookup dictionary
nova_dict = classified.set_index('process')['score'].to_dict()

# Function to compute scores
def compute_scores(processes_str):
    if not isinstance(processes_str, str) or processes_str.strip() == "":
        return pd.Series([np.nan, np.nan])
    processes = processes_str.split("||")
    valid_scores = [nova_dict[p] for p in processes if p in nova_dict]
    if not valid_scores:
        return pd.Series([np.nan, np.nan])
    avg_score = round(np.mean(valid_scores), 3)
    total_score = round(np.sum(valid_scores), 3)
    return pd.Series([avg_score, total_score])

# Apply function to get both average and total scores
recipes[['food_processing_score', 'overall_processing_score']] = recipes['Processes'].apply(compute_scores)

# Save result
output_path = os.path.join(base_path, "recipe_food_processing_scores.csv")
recipes.to_csv(output_path, index=False)

# Display top 10 by overall score
top10 = recipes.sort_values(by='overall_processing_score', ascending=False).head(10)

print("✅ Saved combined results to:", output_path)
print("\n🔝 Top 10 recipes by overall processing score:")
print(top10[['Recipe_id', 'Recipe_title', 'overall_processing_score']])


✅ Saved combined results to: results\recipe_food_processing_scores.csv

🔝 Top 10 recipes by overall processing score:
        Recipe_id                                       Recipe_title  \
96651      127748  5-Cheese Crab Lasagna With Roasted Garlic and ...   
110667     141770                                       Cod Brandade   
101737     132836                                  Vegetable Lasanga   
90439      121531                                   Italian Rum Cake   
104788     135889                   Shell's Potato Soup With Carrots   
33655       64703   Murgh Makhani (Butter Chicken) Restaurant Style!   
61588       92658                                   Cuban Opera Cake   
75625      106707  Belgian Shrimp Croquettes (Croquettes Aux Crev...   
106372     137474                    German Chocolate Cheesecake (2)   
84633      115723                             French-Style Pot Roast   

        overall_processing_score  
96651                      220.0  
110667             

: 